This notebook is used to understand the data, the features and cleanup process

In [ ]:
import pandas as pd
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent))
from src.data.ingestion import DataIngestor
data_path= str((Path.cwd().parent))+"/data/raw/SBAnational.csv"
ingestor = DataIngestor(data_path)
df_clean_clean= ingestor.clean()

In [9]:
df_clean.head(10)

,LoanNr_ChkDgt,Name,City,State,Zip,Bank,BankState,NAICS,ApprovalDate,ApprovalFY,...,LowDoc,ChgOffDate,DisbursementDate,DisbursementGross,BalanceGross,MIS_Status,ChgOffPrinGr,GrAppv,SBA_Appv,is_default
0,1000014003,ABC HOBBYCRAFT,EVANSVILLE,IN,47711,FIFTH THIRD BANK,OH,451120,1997-02-28,1997,...,Y,NaN,28-Feb-99,"$60,000.00",$0.00,P I F,$0.00,"$60,000.00","$48,000.00",0
1,1000024006,LANDMARK BAR & GRILLE (THE),NEW PARIS,IN,46526,1ST SOURCE BANK,IN,722410,1997-02-28,1997,...,Y,NaN,31-May-97,"$40,000.00",$0.00,P I F,$0.00,"$40,000.00","$32,000.00",0
2,1000034009,"WHITLOCK DDS, TODD M.",BLOOMINGTON,IN,47401,GRANT COUNTY STATE BANK,IN,621210,1997-02-28,1997,...,N,NaN,31-Dec-97,"$287,000.00",$0.00,P I F,$0.00,"$287,000.00","$215,250.00",0
3,1000044001,"BIG BUCKS PAWN & JEWELRY, LLC",BROKEN ARROW,OK,74012,1ST NATL BK & TR CO OF BROKEN,OK,0,1997-02-28,1997,...,Y,NaN,30-Jun-97,"$35,000.00",$0.00,P I F,$0.00,"$35,000.00","$28,000.00",0
4,1000054004,"ANASTASIA CONFECTIONS, INC.",ORLANDO,FL,32801,FLORIDA BUS. DEVEL CORP,FL,0,1997-02-28,1997,...,N,NaN,14-May-97,"$229,000.00",$0.00,P I F,$0.00,"$229,000.00","$229,000.00",0
5,1000084002,"B&T SCREW MACHINE COMPANY, INC",PLAINVILLE,CT,6062,"TD BANK, NATIONAL ASSOCIATION",DE,332721,1997-02-28,1997,...,N,NaN,30-Jun-97,"$517,000.00",$0.00,P I F,$0.00,"$517,000.00","$387,750.00",0
6,1000093009,MIDDLE ATLANTIC SPORTS CO INC,UNION,NJ,7083,WELLS FARGO BANK NATL ASSOC,SD,0,1980-06-02,1980,...,N,24-Jun-91,22-Jul-80,"$600,000.00",$0.00,CHGOFF,"$208,959.00","$600,000.00","$499,998.00",1
7,1000094005,WEAVER PRODUCTS,SUMMERFIELD,FL,34491,REGIONS BANK,AL,811118,1997-02-28,1997,...,Y,NaN,30-Jun-98,"$45,000.00",$0.00,P I F,$0.00,"$45,000.00","$36,000.00",0
8,1000104006,TURTLE BEACH INN,PORT SAINT JOE,FL,32456,CENTENNIAL BANK,FL,721310,1997-02-28,1997,...,N,NaN,31-Jul-97,"$305,000.00",$0.00,P I F,$0.00,"$305,000.00","$228,750.00",0
9,1000124001,INTEXT BUILDING SYS LLC,GLASTONBURY,CT,6073,WEBSTER BANK NATL ASSOC,CT,0,1997-02-28,1997,...,Y,NaN,30-Apr-97,"$70,000.00",$0.00,P I F,$0.00,"$70,000.00","$56,000.00",0


We now look at the target column to understand what the ratio of the value counts are and also take care of missing values

In [ ]:
print(df_clean.info())
print(df_clean['MIS_Status'].value_counts(dropna=False))
print(f"PIF/CHGOFF ratio:{df_clean['MIS_Status'].value_counts()['P I F']/(df_clean['MIS_Status'].value_counts()['CHGOFF'])}")

<class 'pandas.DataFrame'>
Index: 897167 entries, 0 to 899163
Data columns (total 28 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   LoanNr_ChkDgt      897167 non-null  int64         
 1   Name               897153 non-null  str           
 2   City               897137 non-null  str           
 3   State              897154 non-null  str           
 4   Zip                897167 non-null  int64         
 5   Bank               895661 non-null  str           
 6   BankState          895654 non-null  str           
 7   NAICS              897167 non-null  int64         
 8   ApprovalDate       897167 non-null  datetime64[us]
 9   ApprovalFY         897167 non-null  str           
 10  Term               897167 non-null  int64         
 11  NoEmp              897167 non-null  int64         
 12  NewExist           897033 non-null  float64       
 13  CreateJob          897167 non-null  int64         
 14  Reta

In [16]:
# Inspect the raw text formatting of the financial columns
print(df_clean[['DisbursementGross', 'GrAppv', 'SBA_Appv']].head(10))
print("\nData types:")
print(df_clean[['DisbursementGross', 'GrAppv', 'SBA_Appv']].dtypes)


  DisbursementGross        GrAppv      SBA_Appv
0       $60,000.00    $60,000.00    $48,000.00 
1       $40,000.00    $40,000.00    $32,000.00 
2      $287,000.00   $287,000.00   $215,250.00 
3       $35,000.00    $35,000.00    $28,000.00 
4      $229,000.00   $229,000.00   $229,000.00 
5      $517,000.00   $517,000.00   $387,750.00 
6      $600,000.00   $600,000.00   $499,998.00 
7       $45,000.00    $45,000.00    $36,000.00 
8      $305,000.00   $305,000.00   $228,750.00 
9       $70,000.00    $70,000.00    $56,000.00 

Data types:
DisbursementGross    str
GrAppv               str
SBA_Appv             str
dtype: object


In [18]:
print(f"Total unique NAICS codes: {df_clean['NAICS'].nunique()}")
print("\nFirst 15 raw values:")
print(df_clean['NAICS'].head(15))

print("\nValue counts of the top 10 codes (checking for placeholders):")
print(df_clean['NAICS'].value_counts().head(10))

Total unique NAICS codes: 1312

First 15 raw values:
0     451120
1     722410
2     621210
3          0
4          0
5     332721
6          0
7     811118
8     721310
9          0
10    811111
11    235950
12    445299
13         0
14         0
Name: NAICS, dtype: int64

Value counts of the top 10 codes (checking for placeholders):
NAICS
0         201667
722110     27941
722211     19435
811111     14539
621210     14034
624410     10092
812112      9180
561730      8873
621310      8718
812320      7880
Name: count, dtype: int64


In [19]:
# Convert to datetime using mixed format to safely catch variations
approval_dates = pd.to_datetime(df_clean['ApprovalDate'], format='mixed')

print("--- TEMPORAL PROFILE ---")
print(f"Earliest Loan Approval: {approval_dates.min()}")
print(f"Latest Loan Approval:   {approval_dates.max()}")

print("\nMissing dates count:")
print(approval_dates.isna().sum())

--- TEMPORAL PROFILE ---
Earliest Loan Approval: 1976-01-09 00:00:00
Latest Loan Approval:   2075-12-30 00:00:00

Missing dates count:
0
